# Companion Lab 01 — Train a Drug-Synergy Neural Network in PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/5x5x5x5/taihls/blob/curriculum-v2/notebooks/01-pytorch-neural-net.ipynb)

In **Chapter 3.2**, we trained a tiny 25-parameter neural network from scratch in NumPy using calculus-free downhill peeking. Here, we scale up to **PyTorch** — where `loss.backward()` computes exact downhill slopes for thousands of weights simultaneously via backpropagation — on the exact same **two-drug curved therapeutic window**.

> **Tip:** On Colab, you can enable a GPU via *Runtime → Change runtime type → T4 GPU* (this small network also trains in seconds on CPU).

In [ ]:
!pip install -q torch scikit-learn matplotlib numpy

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Running on device:', device)

## 1. Recreate the curved two-drug therapeutic window

**Predict before running:** Why can't a straight-line model separate a ring/arch-shaped safe zone from ineffective/toxic doses around it?

In [ ]:
rng = np.random.default_rng(7)
n = 500
drug_a = rng.uniform(-2.0, 2.0, size=n)
drug_b = rng.uniform(-2.0, 2.0, size=n)
X = np.column_stack([drug_a, drug_b]).astype(np.float32)

radius = np.sqrt(drug_a**2 + (drug_b + 0.35)**2)
effective = (radius > 0.65) & (radius < 1.55) & (drug_b > -1.0)
y = np.where(rng.random(n) < 0.04, ~effective, effective).astype(np.float32)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30, random_state=0, stratify=y)

X_tr_t = torch.tensor(X_tr, device=device)
y_tr_t = torch.tensor(y_tr, device=device).unsqueeze(1)
X_te_t = torch.tensor(X_te, device=device)
y_te_t = torch.tensor(y_te, device=device).unsqueeze(1)
print(f'Train: {len(X_tr)} assays | Test: {len(X_te)} assays')

## 2. Build and train a two-hidden-layer ReLU network in PyTorch

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Total learnable parameters: {n_params}')

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)

tr_hist, te_hist = [], []
for epoch in range(250):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(X_tr_t), y_tr_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        te_loss = loss_fn(model(X_te_t), y_te_t).item()
    tr_hist.append(loss.item())
    te_hist.append(te_loss)
    if epoch % 50 == 0:
        print(f'Epoch {epoch:>3} | train loss = {loss.item():.3f} | test loss = {te_loss:.3f}')

with torch.no_grad():
    probs = torch.sigmoid(model(X_te_t)).cpu().numpy().ravel()
acc = ((probs >= 0.5) == y_te).mean()
print(f'\nHeld-out test accuracy: {acc:.1%}')

## 3. Visualize learning curves and the learned curved therapeutic boundary

In [ ]:
BLUE, VERMILION = '#0072B2', '#D55E00'
xx, yy = np.meshgrid(np.linspace(-2.1, 2.1, 240), np.linspace(-2.1, 2.1, 240))
grid_t = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32, device=device)
with torch.no_grad():
    zz = torch.sigmoid(model(grid_t)).cpu().numpy().reshape(xx.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4))
ax1.plot(tr_hist, lw=2.2, color=BLUE, label='Train loss')
ax1.plot(te_hist, lw=2.2, color=VERMILION, ls='--', label='Test loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Binary cross-entropy loss')
ax1.set_title('A. PyTorch training & test loss'); ax1.legend(frameon=False); ax1.grid(alpha=0.25)

ax2.contourf(xx, yy, zz, levels=np.linspace(0, 1, 11), cmap='RdBu_r', alpha=0.55)
ax2.contour(xx, yy, zz, levels=[0.5], colors=['#111111'], linewidths=2.0)
for lbl, col, name in [(0, BLUE, 'Ineffective / Toxic'), (1, VERMILION, 'Therapeutic window')]:
    m = y_te == lbl
    ax2.scatter(X_te[m, 0], X_te[m, 1], s=22, color=col, edgecolors='white', lw=0.4, label=name)
ax2.set_xlabel('Drug A dose (std)'); ax2.set_ylabel('Drug B dose (std)')
ax2.set_title(f'B. Learned boundary (test acc = {acc:.1%})'); ax2.legend(frameon=False, loc='lower left')
plt.tight_layout()
plt.show()

## 4. Modify & Make (PRIMM challenges)

1. **Remove ReLU:** Comment out both `nn.ReLU()` lines and re-run. Watch the curved arch collapse straight back into a single diagonal line!
2. **Overfit on purpose:** Shrink `n = 80` and enlarge the hidden layers to `128` neurons each, training for `800` epochs. Watch `te_hist` (test loss) start climbing while `tr_hist` hits `0.000`!
3. **Learning-rate cliff:** Try `lr = 0.0005` vs. `lr = 0.5` and compare the loss curves.